# Dataset preparation for code-completion

In [1]:
def default_params(): 
    return {
        'current_model': 'M1',
        'gpu': True,
        'quantization': 'none', #['none',"int4", "int8", "float32", "float16"]
        'dataset': {
            'path': '/workspaces/CodeSmells/semeru-datasets/code_smells/generation/curated_500.json',
            'prompt_column': 'prompt',
            'content_column' : 'code',
            'sampling_size': 500,
            'prompt_text' : """complete the following incomplete Python function:\n"""
        },
        'completion_extra_limit': 100,
        'output_generation_dir': '/workspaces/CodeSmells/datax/code_smells/generation/dataset',
        'decoding_strategies' : ['greedy_search', 'beam_search', 'sampling', 'contrastive_search', 'top_k_sampling', 'top_p_sampling'],
        'cache_dir': '/workspaces/CodeSmells/datax/hugging_face_cache',
        'causal_models': {
            'M1' : 'codellama/CodeLlama-7b-hf', #https://huggingface.co/codellama/CodeLlama-7b-hf, 
            'M2' : 'mistralai/Mistral-7B-v0.3', #https://huggingface.co/mistralai/Mistral-7B-v0.3,
            'M3' : 'microsoft/Phi-3.5-mini-instruct', #https://huggingface.co/microsoft/Phi-3.5-mini-instruct 
            'M4' : 'Qwen/Qwen2.5-Coder-7B', #https://huggingface.co/Qwen/Qwen2.5-Coder-7B
            'M5' : 'facebook/incoder-6B', #https://huggingface.co/facebook/incoder-6B
            'M6' : 'bigcode/starcoder2-7b', #https://huggingface.co/bigcode/starcoder2-7b 
            'M7' : 'deepseek-ai/DeepSeek-R1-Distill-Llama-8B', #https://huggingface.co/deepseek-ai/DeepSeek-R1-Distill-Llama-8B
            'M8' : 'deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B', #https://huggingface.co/deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B
        },
    }
params = default_params()


### Imports

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns; sns.set_theme()
from collections import Counter
import plotly.express as px
from transformers import AutoTokenizer
import torch
import os
import gc

In [3]:
from transformers import LlamaForCausalLM, CodeLlamaTokenizer
from datasets import load_dataset

#### GPU

In [4]:
! nvidia-smi

Mon Feb 24 19:40:05 2025       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 470.103.01   Driver Version: 470.103.01   CUDA Version: 12.3     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  NVIDIA A100-PCI...  Off  | 00000000:61:00.0 Off |                    0 |
| N/A   37C    P0    34W / 250W |      0MiB / 40536MiB |      0%      Default |
|                               |                      |             Disabled |
+-------------------------------+----------------------+----------------------+
                                                                               
+-------

In [5]:
torch.__version__

'2.1.2+cu121'

In [6]:
device = torch.device("cuda:0" if torch.cuda.is_available() and params['gpu'] else "cpu")
device

device(type='cuda', index=0)

In [7]:
torch.cuda.memory_allocated()

0

### Model loading

In [8]:
def instantiate_llm(model_name:str, cache_dir:str):
     '''Instantiate AutoModelForCausalLM'''
     tokenizer = CodeLlamaTokenizer.from_pretrained(model_name, cache_dir = cache_dir)
     model = None
     if params['quantization'] == 'int4':
          model = LlamaForCausalLM.from_pretrained(model_name, cache_dir = cache_dir, load_in_4bit=True)
     elif params['quantization'] == 'int8':
          model = LlamaForCausalLM.from_pretrained(model_name, cache_dir = cache_dir, load_in_8bit=True)
     elif params['quantization'] == 'float32':
          model = LlamaForCausalLM.from_pretrained(model_name, cache_dir = cache_dir, torch_dtype=torch.float32)
     elif params['quantization'] == 'float16':
          model = LlamaForCausalLM.from_pretrained(model_name, cache_dir = cache_dir, torch_dtype=torch.float16)
     else: 
          model = LlamaForCausalLM.from_pretrained(model_name, cache_dir = cache_dir)

     return tokenizer, model

In [9]:
tokenizer, pretrained_model = instantiate_llm(params['causal_models'][params['current_model']], params['cache_dir'])

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [10]:
pretrained_model.config

LlamaConfig {
  "_name_or_path": "codellama/CodeLlama-7b-hf",
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 1,
  "eos_token_id": 2,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 11008,
  "max_position_embeddings": 16384,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 32,
  "num_key_value_heads": 32,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_scaling": null,
  "rope_theta": 1000000,
  "tie_word_embeddings": false,
  "torch_dtype": "bfloat16",
  "transformers_version": "4.36.2",
  "use_cache": true,
  "vocab_size": 32016
}

In [11]:
pretrained_model.to(device) #WARNING, Verify the device before assigning to memory

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32016, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaSdpaAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (v_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=4096, out_features=11008, bias=False)
          (up_proj): Linear(in_features=4096, out_features=11008, bias=False)
          (down_proj): Linear(in_features=11008, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm()
        (post_attention_layernorm): LlamaRMSNorm()
      )
    )
    (norm): LlamaRMSNorm()
  )
  (lm_head):

### Load dataset

In [12]:
completion_df = pd.read_json(params['dataset']['path'])

In [13]:
completion_df = completion_df[:5]

### Complete prompts

In [14]:
def generate_text(prompt, decoding_strategy: str, max_length=None):
    # Tokenize the input prompt
    # Tokenize the input prompt with padding and attention mask
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    input_ids = inputs.input_ids
    attention_mask = inputs.attention_mask

    # Choose the decoding strategy
    if decoding_strategy == "greedy_search":
        output_ids = pretrained_model.generate(input_ids, attention_mask=attention_mask, max_length=max_length, pad_token_id=tokenizer.pad_token_id)
    elif decoding_strategy == "beam_search":
        output_ids = pretrained_model.generate(input_ids, attention_mask=attention_mask, max_length=max_length, num_beams=5, early_stopping=True, pad_token_id=tokenizer.pad_token_id)
    elif decoding_strategy == "sampling":
        output_ids = pretrained_model.generate(input_ids, attention_mask=attention_mask, max_length=max_length, do_sample=True, top_k=50, top_p=0.9, pad_token_id=tokenizer.pad_token_id)
    elif decoding_strategy == "contrastive_search":
        output_ids = pretrained_model.generate(input_ids, attention_mask=attention_mask, max_length=max_length, penalty_alpha=0.6, top_k=4, pad_token_id=tokenizer.pad_token_id)
    elif decoding_strategy == "top_k_sampling":
        output_ids = pretrained_model.generate(input_ids, attention_mask=attention_mask, max_length=max_length, do_sample=True, top_k=50, pad_token_id=tokenizer.pad_token_id)
    elif decoding_strategy == "top_p_sampling":
        output_ids = pretrained_model.generate(input_ids, attention_mask=attention_mask, max_length=max_length, do_sample=True, top_p=0.92, pad_token_id=tokenizer.pad_token_id)
    else:
        raise ValueError(f"Unsupported decoding strategy: {decoding_strategy}")


    # Decode the generated text
    generated_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)

    return generated_text

In [15]:
def complete_prompts(dataframe):
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token  # Set pad token to eos_token if not set
    prompt_lenght = len(tokenizer.encode(params['dataset']['prompt_text'], add_special_tokens=False))
    updated_dataframe = dataframe.copy()
    for decoding_strategy in params['decoding_strategies']:
        updated_dataframe[decoding_strategy] = updated_dataframe.apply(lambda row: generate_text(row[params['dataset']['prompt_column']], decoding_strategy ,len(tokenizer.encode(row[params['dataset']['content_column']], add_special_tokens=True)) + prompt_lenght+ params['completion_extra_limit']), axis=1)
        #updated_dataframe[decoding_strategy] = updated_dataframe.apply(lambda row: generated_text(row[params['dataset']['prompt_column']], decoding_strategy), axis=1)
        updated_dataframe[decoding_strategy] = updated_dataframe[decoding_strategy].map(lambda completed_code: completed_code[len(params['dataset']['prompt_text']):])
    return updated_dataframe

In [16]:
completed_df = complete_prompts(completion_df)

### Store dataset 

In [17]:
def create_folder(path):
    if not os.path.exists(path):
        os.makedirs(path)

In [18]:
output_generation_dir = f"{params['output_generation_dir']}/{params['current_model']}_q_{params['quantization']}"
create_folder(output_generation_dir)
completed_df.to_json(f"{output_generation_dir}/curated_generation_{params['dataset']['sampling_size']}.json")

### Clean

In [ ]:
del pretrained_model
torch.cuda.empty_cache()
gc.collect()

185

: 